# X3DNA Fiber

This notebook demonstrates building idealized canonical nucleic-acid duplexes with X3DNA's `fiber` program. Given a base sequence for one strand, `fiber` constructs the corresponding canonical (Arnott) A-DNA, B-DNA, or Z-DNA helix, or an A-form RNA duplex, generating the complementary strand automatically. The result is a fully built 3D structure (atomic coordinates) for the requested helix form.

Learn more at [x3dna.org](https://x3dna.org).

**Note:** X3DNA is a user-provisioned local install (distributed under CC-BY-NC-4.0). It is not bundled with proto-tools. To run this tool you must install X3DNA v2.4 locally and either set the `X3DNA` environment variable to the install root (the directory containing `bin/fiber`) or pass `x3dna_dir` in the config. The basic and advanced usage cells below will error without a local X3DNA install.

In [ ]:
from proto_tools.tools import run_x3dna_fiber, X3DNAFiberInput, X3DNAFiberConfig

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("x3dna")
display_overview("x3dna")
display_docs_section("x3dna", "Background")

## Available tools

In [ ]:
display_available_tools("x3dna")

### `run_x3dna_fiber`

`run_x3dna_fiber` builds an idealized fiber nucleic-acid duplex for each input sequence using X3DNA's `fiber` program. The `form` config selects the canonical model (A-DNA, B-DNA, Z-DNA, or A-form RNA), the complementary strand is generated automatically, and the result is one fully built `Structure` per input sequence. T and U bases are interconverted to match the requested form (T for DNA, U for RNA).

**Note:** The cells below require a local X3DNA v2.4 install (`bin/fiber`) with the `X3DNA` environment variable set (or `x3dna_dir` passed in the config). They will error without it.

In [ ]:
# Display input API reference
display_api_reference("x3dna-fiber", "input", "run_x3dna_fiber")

# A realistic operator-like sequence (one strand, 5'->3'); the complement is generated.
inputs = X3DNAFiberInput(sequences=["GGGCAAAATGCACTGCACTTTGGG"])

In [ ]:
# Display config API reference
display_api_reference("x3dna-fiber", "config", "run_x3dna_fiber")

# Build a canonical B-form DNA duplex (the default form).
config = X3DNAFiberConfig(form="B-DNA")

In [ ]:
# Display output API reference
display_api_reference("x3dna-fiber", "output", "run_x3dna_fiber")

# Run fiber generation
result = run_x3dna_fiber(inputs, config)

# Inspect the built duplex
print(f"Number of structures: {len(result.structures)}")
duplex = result.structures[0]
print(f"Chain IDs:    {duplex.get_chain_ids()}")
print(f"Residue count: {duplex.num_residues}")

## Advanced usage

The `form` config selects the canonical helix model. Use `form="RNA"` for an A-form RNA duplex (T is converted to U automatically), `form="Z-DNA"` for the left-handed Z helix (best suited to alternating purine/pyrimidine, GC-rich sequences), and `single_stranded=True` to emit only the sense strand instead of the full duplex.

**Note:** These cells also require a local X3DNA install and will error without it.

In [ ]:
# A-form RNA duplex from an RNA sequence (T/U interconverted to match the form).
rna_inputs = X3DNAFiberInput(sequences=["GGCGAGCUUCGCUCGCC"])
rna_config = X3DNAFiberConfig(form="RNA")
rna_result = run_x3dna_fiber(rna_inputs, rna_config)
rna_duplex = rna_result.structures[0]
print(f"RNA duplex chains:    {rna_duplex.get_chain_ids()}")
print(f"RNA duplex residues:  {rna_duplex.num_residues}")

In [ ]:
# Left-handed Z-DNA from an alternating GC sequence.
z_inputs = X3DNAFiberInput(sequences=["CGCGCGCGCGCG"])
z_config = X3DNAFiberConfig(form="Z-DNA")
z_result = run_x3dna_fiber(z_inputs, z_config)
z_duplex = z_result.structures[0]
print(f"Z-DNA chains:    {z_duplex.get_chain_ids()}")
print(f"Z-DNA residues:  {z_duplex.num_residues}")

In [ ]:
# Single-stranded output: emit only the sense strand instead of the full duplex.
ss_inputs = X3DNAFiberInput(sequences=["GGGCAAAATGCACTGCACTTTGGG"])
ss_config = X3DNAFiberConfig(form="B-DNA", single_stranded=True)
ss_result = run_x3dna_fiber(ss_inputs, ss_config)
ss_structure = ss_result.structures[0]
print(f"Single-strand chains:    {ss_structure.get_chain_ids()}")
print(f"Single-strand residues:  {ss_structure.num_residues}")

## Export Results

The generated structures can be exported to PDB or CIF format for downstream analysis or visualization. Each structure is written to `structure_{i}.{ext}` under the export path.

In [ ]:
from pathlib import Path

# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export the built structures to PDB
result.export("x3dna_fiber", export_path=output_dir, file_format="pdb")